# 剪枝（结构化剪枝）学习笔记本 notebook 只注重「思路 + 重要代码」，**不带运行输出**。每个流程按 **原理 / 重要 bash 命令 / 重要代码** 三部分整理。代码均取自 `scripts/`，标注了来源文件，可回原脚本看完整实现。

## 一、通用准备（所有实验共用，只讲一次）

### 1. 环境安装（ultralytics + torch-pruning）**原理**两个核心库：- `ultralytics`：YOLO11 官方实现，负责模型加载、训练、验证。- `torch-pruning`：结构化剪枝库，核心是 **DependencyGraph（依赖图）**。**名词**- **结构化剪枝 vs 非结构化剪枝**：结构化剪枝删「整条卷积输出通道」，体积/计算量/速度同时受益；非结构化只把单个权重置零，几乎不加速。本项目做的是结构化。- **依赖图（DependencyGraph）**：记录层与层之间的通道耦合。剪掉某层 conv 的输出通道时，下游 BN 通道数、相连 conv 的输入通道、C2f 的 split/concat 都要跟着一起删，否则形状对不上会报错。`torch-pruning` 的 `build_dependency` + `get_pruning_group` 就是干这个。本机环境：RTX 5060 Ti、PyTorch 2.11.0+cu128、torch-pruning 1.6.1。

**重要 bash 命令**```bash# 建虚拟环境（可选）python -m venv .venv# 安装依赖pip install ultralytics torch torchvisionpip install torch-pruning# 验证安装python -c "import ultralytics, torch_pruning, torch; print(ultralytics.__version__, torch_pruning.__version__, torch.__version__)"```

In [ ]:
# 科研里反复用到的几个入口（来自各脚本顶部 import）import torchimport torch_pruning as tp                         # 依赖图 / 剪枝 API / count_ops_and_paramsfrom ultralytics import YOLO                       # 模型加载、train/val 入口from ultralytics.nn.modules import C2f, C2PSA      # YOLO11 的 CSP 模块，剪枝保护规则要用from ultralytics.models.yolo.detect.train import DetectionTrainer  # 微调时绕过 get_model 重建

### 2. 数据下载与划分（coco8 / coco128 / coco2017）**原理**数据规模三档：- `coco8`：8 张，只用于验证流程/环境是否跑通。- `coco128`：128 张，小规模剪枝实验用（训练快，适合快速试方法）。- `coco2017`：约 11.8 万训练图 + 5000 验证图，正式实验用。**为什么 coco128 要自己划分**：coco128 本身没有官方 train/val 划分，脚本按 8:2（102 训练 / 26 验证）固定随机种子 42 切分，保证实验可复现。**名词**：训练集（更新权重）/ 验证集（只测不练，用来评估精度、选模型）。注意：coco2017 数据量大，放在仓库外的 `C:/Users/22565/datasets/coco`，不进 OneDrive。

**重要 bash 命令**```bash# coco8 / coco128：ultralytics 首次 train/val 时会自动下载# 划分 coco128（生成 datasets/coco128_split）python scripts/split_coco128.py# coco2017：见 configs/coco2017.yaml 里的 download 段，或手动下载到 C:/Users/22565/datasets/coco```

In [ ]:
# 来源：scripts/split_coco128.py —— 固定种子 8:2 切分import randomfrom pathlib import Pathrandom.seed(42)images = sorted(Path("datasets/coco128/images/train2017").glob("*.jpg"))random.shuffle(images)cut = int(len(images) * 0.8)          # 8:2train, val = images[:cut], images[cut:]

### 3. 模型下载与加载（yolo11n.pt / yolo11s.pt）**原理**- **YOLO11n / YOLO11s**：n = nano（约 2.6M 参数），s = small（约 9.5M 参数）。小模型冗余本来就少，剪枝收益有限——这是本项目「剪得少、掉点多」的根本原因。- **.pt 里有什么**：ultralytics 的 `.pt` 是一个 dict，含 `model`（网络结构+权重）、`ema`、`optimizer`、`train_args` 等；`YOLO(path)` 负责解析成可用的检测模型。- **预训练权重**：在 COCO 上已训练好的权重，作为微调/剪枝的起点，比从零训练快很多。

**重要 bash 命令**```bash# 官方预训练权重放到 weights/ 或 models/，ultralytics 也能自动下载# 也可用脚本触发下载：python -c "from ultralytics import YOLO; YOLO('yolo11n.pt')"```

In [ ]:
# 来源：scripts/run_yolo11s_coco2017_baseline.py —— 下载 + 加载from ultralytics import YOLOfrom ultralytics.utils.downloads import attempt_download_assetweights = Path("weights/yolo11s.pt")downloaded = Path(attempt_download_asset(weights))   # 不存在则自动下载并返回路径model = YOLO(str(downloaded))                        # 解析 .pt，拿到检测模型

### 4. 训练与监控进度（train 参数、results.csv / png、best.pt）**原理**核心 `train` 参数：- `epochs`：训练轮数。- `imgsz`：输入尺寸（常见 640，本项目 coco2017 训练用 512）。- `batch`：批大小。- `device`：用哪张卡（0 = 第一张 GPU）。- `seed`：随机种子，固定后可复现。- `patience`：早停——连续 N 个 epoch 无提升就提前停。**best.pt 怎么来**：ultralytics 每个 epoch 在验证集上算 `fitness`（综合 mAP/recall 的分数），最高那轮存为 `best.pt`，最后一轮存 `last.pt`。**results.csv / results.png**：训练曲线，记录每个 epoch 的 loss 和指标，用来判断有没有过拟合/训练是否正常。

**重要 bash 命令**```bash# 命令行方式（coco128 上微调 YOLO11n）yolo detect train data=configs/coco128_split.yaml model=models/yolo11n.pt epochs=3 imgsz=640 batch=8 device=0# 或跑仓库脚本（coco2017 全量训练）python scripts/train_yolo11s_coco2017.py```

In [ ]:
# 来源：scripts/train_yolo11s_coco2017.py —— 关键 train 设置from ultralytics import YOLOmodel = YOLO("weights/yolo11s.pt")model.train(    data="configs/coco2017.yaml", epochs=30, imgsz=512, batch=64,    device=0, workers=2, seed=42, deterministic=False, patience=10,    project="runs/train/xxx", name="时间戳", exist_ok=True, plots=True,)

### 5. 结果指标含义（P / R / mAP50 / mAP50-95 / GMACs / 参数 / FLOPs）**原理**（老师必问，重点）- **Precision / Recall**：精确率 = 预测出的框里有多少是真框；召回率 = 真框里有多少被找到。二者是 trade-off。- **mAP50 / mAP50-95**：在 IoU 阈值 0.5（或 0.5~0.95 取平均）下算的平均精度。mAP50 宽松，**mAP50-95 更严格、是主指标**。- **参数量 (params)**：所有权重个数，影响体积和显存。- **GMACs vs FLOPs**：都是计算量单位。**1 GMAC = 2 FLOPs**（一次乘加 MAC 算 2 次浮点运算）。仓库统一用 GMACs。- **为什么 FLOPs 降了但速度不一定快**：减的量太少（本项目只有 ~1–4%）时，测速波动比收益还大；且真实推理还受内存带宽、算子调度影响，FLOPs 只是近似。

**重要 bash 命令**```bash# 验证并输出指标（用 yolo 命令行）yolo detect val data=configs/coco128_split.yaml model=runs/train/experiment02_coco128/weights/best.pt imgsz=640 device=0```

In [ ]:
# 取验证指标（来源：scripts/run_yolo11s_coco2017_baseline.py）metrics = model.val(data=..., split="val", imgsz=640, batch=32, device=0)map50     = float(metrics.box.map50)   # mAP50map50_95  = float(metrics.box.map)     # mAP50-95precision = float(metrics.box.mp)      # Precisionrecall    = float(metrics.box.mr)      # Recallinfer_ms  = float(metrics.speed["inference"])   # 每张图验证推理耗时(ms)# 统计 GMACs 和参数量（来源：scripts/prune_independent_compare.py）import torch_pruning as tpmacs, _ = tp.utils.count_ops_and_params(model, example_inputs=torch.zeros(1, 3, 640, 640))gmacs = macs / 1e9                       # 除以 1e9 得到 GMACsparams = sum(p.numel() for p in model.parameters())

## 二、各实验专属流程

### 敏感度分析（实验 03 coco128 / 实验 09 coco2017）**原理**- **目的**：找出哪些层的通道「剪了不疼」，为后续结构化剪枝筛候选层。- **方法（masking 置零）**：每次从同一个 best.pt 重载，把某一层 L1 重要性最低的约 10% 输出通道权重**置零（不是真删）**，在验证集上测精度下降量 `drop`。- **L1 重要性**：卷积核权重绝对值越大越重要，用 `weight.abs().mean()` 给每个输出通道打分。- **关键区别**：masking 只是置零估趋势，**不减少参数量/GMACs**，也不等价于真剪枝（真剪会删通道、改结构）。所以它是「启发式筛选」，不是精确预测。**名词**：敏感度 = 屏蔽该层后 mAP 下降越多，越敏感、越不能剪。

**重要 bash 命令**```bashpython scripts/prune_sensitivity.py \    --weights runs/train/experiment02_coco128/weights/best.pt \    --data configs/coco128_split.yaml --ratio 0.10 \    --output reports/experiment03_sensitivity.csv```

In [ ]:
# 来源：scripts/prune_sensitivity.py —— 候选层 + L1 置零def find_candidate_layers(model):    # 全部输出通道 >= 16 的卷积层    return [(n, m) for n, m in model.model.named_modules()            if isinstance(m, torch.nn.Conv2d) and m.out_channels >= 16]def mask_low_l1_filters(conv, ratio):    count = max(1, round(conv.out_channels * ratio))       # 要屏蔽的通道数    count = min(count, conv.out_channels - 1)    scores = conv.weight.detach().abs().flatten(1).mean(1)  # L1：每个输出通道权重绝对值均值    indices = torch.argsort(scores)[:count]                 # 取最小的一批    with torch.no_grad():        conv.weight[indices] = 0                            # 置零（不是删除）        if conv.bias is not None:            conv.bias[indices] = 0    return count

### 独立法结构化剪枝（实验 04）**原理**- **独立法**：light/balanced/strong 三组分别从**同一个 best.pt 独立**开始，剪不同数量的低敏感层（3/6/9 个），互不影响。- **真正的结构化剪枝**：这次真的删通道——`DependencyGraph` 自动把下游 BN、相连 conv 输入通道、深度卷积宽度一起删，保证形状对齐。- **候选层筛选（read_candidates）**：从敏感度 CSV 里挑「低敏感（mAP50-95 drop ≤ 0.005）+ 输出通道 ≥ 64 + 排除首层/注意力/检测头」的层。- **choose_indices**：按 L1 打分选要删的通道，并把数量**对齐到 8 的倍数**（channel_multiple=8），硬件对 8 的倍数通道更友好。- **微调**：剪完用 `finetune_pruned_compare.py` 恢复精度（见贪心块里的微调代码）。

**重要 bash 命令**```bashpython scripts/prune_independent_compare.py \    --weights runs/train/experiment02_coco128/weights/best.pt \    --data configs/coco128_split.yaml \    --sensitivity reports/experiment03_sensitivity.csv \    --profiles light balanced strong```

In [ ]:
# 来源：scripts/prune_independent_compare.py —— 选通道 + 真剪核心def choose_indices(conv, ratio, channel_multiple):    desired = max(1, round(conv.out_channels * ratio))    if conv.out_channels >= channel_multiple * 2:          # 对齐到 8 的倍数        desired = max(channel_multiple, round(desired / channel_multiple) * channel_multiple)        desired = min(desired, conv.out_channels - channel_multiple)    scores = conv.weight.detach().abs().flatten(1).mean(1)  # L1 打分    return torch.argsort(scores)[:desired].cpu().tolist()   # 最低的 desired 个def prune_one_layer(model, layer_name, ratio, channel_multiple, example):    for p in model.parameters():        p.requires_grad_(True)     # 关键：ultralytics 加载后 requires_grad=False，                                   # 不设 True 依赖图会追踪不到可剪模块    conv = dict(model.named_modules())[layer_name]    indices = choose_indices(conv, ratio, channel_multiple)    graph = tp.DependencyGraph().build_dependency(model, example_inputs=example)   # 建依赖图    group = graph.get_pruning_group(conv, tp.prune_conv_out_channels, idxs=indices) # 拿到联动组    if not graph.check_pruning_group(group):        raise RuntimeError("依赖图拒绝剪枝")    group.prune()                 # 一次性删掉 conv + 下游所有联动通道

### 贪心结构化剪枝（实验 05 / 10）**原理**- **贪心搜索**：每一步在所有候选层上「试剪」，选**代价最小**的一步接受。代价 = mAP 下降 / GMAC 减少（每省一点计算量掉多少精度）。每步都在当前已剪模型上重新算 L1、重新选，允许同一层重复剪。- **与独立法的区别**：独立法一次性剪多个层；贪心一步一步来，每步只接受当前最优。- **保护规则（safe_prune）**：不剪首层(stem)、注意力层、C2f/C2PSA 的 CSP 分块宽度、检测头最终输出（DFL/类别框语义）；每层至少保留初始通道一半。- **微调恢复（run_finetune）**：剪完精度掉，用少量轮次微调补回来。两个关键点：① 直接 `trainer.model = pruned.model`，绕过 ultralytics 按 YAML 重建（否则剪掉的结构会被重建回去）；② 固定 BN（统计量和仿射参数都不更新），因为小数据会把 BN 统计冲坏。**名词**：backward_check = 剪完做前向+反向检查，确认梯度有限、结构没坏才接受这一步。

**重要 bash 命令**```bash# coco128 贪心python scripts/prune_greedy.py --device 0 --max-steps 20# coco2017 贪心python scripts/prune_greedy_coco2017.py --device 0 --max-steps 20 --target-reduction 0.20```

In [ ]:
# 来源：scripts/prune_greedy.py —— 保护规则 + 代价函数split_outputs = {m.cv1.conv for m in model.modules() if isinstance(m, (C2f, C2PSA))}for dep, idxs in group:    module_name = reverse[dep.target.module]    out_prune = graph.is_out_channel_pruning_fn(dep.handler)    if module in split_outputs and out_prune:        raise ValueError(f"protect_CSP_chunk_width: {module_name}")    if module_name == "model.0" or module_name.startswith(("model.0.", "model.10.")):        raise ValueError(f"protect_stem_or_attention: {module_name}")    if module_name.startswith("model.23.dfl"):        raise ValueError(f"protect_DFL: {module_name}")# 贪心的「代价」：每省 1% GMAC 掉多少 mAP（越小越好）score = drop / gain   # drop = 当前步 mAP 下降；gain = 相对 baseline 的 GMAC 减少

In [ ]:
# 来源：scripts/greedy_evaluation.py —— 微调恢复：直塞模型 + 固定 BNfrom ultralytics.models.yolo.detect.train import DetectionTrainerclass FrozenBNTrainer(DetectionTrainer):    def _model_train(self):        super()._model_train()          # 先让父类把模型置为 train()        for m in self.model.modules():            if isinstance(m, nn.BatchNorm2d):                m.eval()                # 固定 BN：小数据微调别让统计量被冲坏                for p in m.parameters():                    p.requires_grad_(False)trainer = FrozenBNTrainer(overrides=overrides)trainer.model = pruned.model            # 关键：直接塞已剪模型，绕过 get_model() 按 YAML 重建trainer.train()

### 梯度分档剪枝（实验 06）**原理**- **梯度重要性（Taylor 类）**：`mean(|W × ∂L/∂W|)`——权重 × 权重梯度 的绝对值。直观理解：这个通道对损失函数的贡献/敏感度，比纯 L1 幅值更能反映「剪掉后损失会怎么变」。- **分档（tiered）**：复用实验 03 的 L1 masking 敏感度，把层分成 low(≤0.005) / medium(0.005~0.020] / high(>0.020) 三档，A–E 五组按不同比例剪（如 A = 低 12.5%/中 0/高 0，E = 低 25%/中 15%/高 5%）。- **collect_scores 只统计不改权重**：过一遍全部训练图算梯度打分，但**不更新权重、固定 BN**，并验证模型张量前后一致（fingerprint 相同）。

**重要 bash 命令**```bashpython scripts/prune_gradient_tiered.py --device 0 --epochs 10 --max-map-drop 0.02```

In [ ]:
# 来源：scripts/prune_gradient_tiered.py —— 梯度打分核心loss, _ = model(batch)loss = loss.sum() / sizeloss.backward()for name, total in scores.items():    weight = modules[name].weight    value = (weight.detach() * weight.grad.detach()).abs().flatten(1).mean(1)  # |W × ∇L|    total.add_(value.double().cpu(), alpha=size)   # 累加，最后除以总图数取平均

### 测速与计算量统计（GMACs / 参数 / 前向耗时）**原理**- **GMACs / 参数**：用 torch-pruning 的 `count_ops_and_params` 在**未融合(unfused)**结构上统计，保证剪枝前后口径一致。- **测速**：`fuse()` 融合 Conv+BN（推理常用、更快），batch=1、640×640，预热 30 次后测量 250 次取**中位数**（抗偶发波动）。用 CUDA Event 计时 + 同步。- **为什么「FLOPs 降了但没加速」**：减的量太少（~1–4%）时，轮间波动比收益还大；真实推理还受内存带宽/算子调度影响。**名词**：fuse（融合）= 把 Conv 后的 BN 合并进 Conv 权重，推理时少一层计算。

**重要 bash 命令**```bash# 测速和 GMACs 已内嵌在剪枝脚本里（benchmark / stats），一般无需单独跑```

In [ ]:
# 来源：scripts/greedy_evaluation.py —— 测速核心：融合 + 预热 + CUDA Event + 中位数model = YOLO(path).model.float().eval().fuse()   # 融合 Conv+BNexample = torch.zeros(1, 3, 640, 640, device=device)with torch.inference_mode():    for _ in range(30):                          # 预热 30 次        model(example)    torch.cuda.synchronize()    start, end = torch.cuda.Event(True), torch.cuda.Event(True)    start.record(); model(example); end.record()    torch.cuda.synchronize()    elapsed = start.elapsed_time(end)            # 单次前向耗时(ms)# 多轮取 250 次的中位数作为 latency_median_ms